# Week 1 Project - Predicting p-factor (RBC/PNC)

## 1. Imports

In [2]:
from rbclib import RBCPath
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.error import HTTPError

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

from ipywidgets import IntProgress
from IPython.display import display

GROUP_NAME = "week1proj-europe"
OUTPUT_PATH = f"results/{GROUP_NAME}.tsv"
STABILITY_SEEDS = [1, 7, 42, 99]

FINAL_FEATURES = [
    'global_mean_thickness',
    'total_cortical_volume',
    'age',
    'age_sq',
    'sex_numeric',
    'parent_edu_avg'
]

print(f"Group: {GROUP_NAME}")
print(f"Output: {OUTPUT_PATH}")

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Group: week1proj-europe
Output: results/week1proj-europe_final.tsv


## 2. Load Participant Metadata

In [3]:
rbcdata_path = Path('/home/jovyan/shared/data/RBC')

with (rbcdata_path / 'train_participants.tsv').open('r') as f:
    train_data = pd.read_csv(f, sep='\t')
with (rbcdata_path / 'test_participants.tsv').open('r') as f:
    test_data = pd.read_csv(f, sep='\t')

all_data = pd.concat([train_data, test_data])

print(f"Train: {len(train_data)} | Test: {len(test_data)} | Total: {len(all_data)}")
train_data.head()

Train: 1280 | Test: 321 | Total: 1601


,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,p_factor
0,1000393599,PNC,PNC1,PNC1,1,15.583333,Male,Black,not Hispanic or Latino,22.15,Right,9th Grade,Complete primary,Complete secondary,0.589907
1,1000881804,PNC,PNC1,PNC1,1,14.916667,Male,Black,not Hispanic or Latino,21.52,Right,7th Grade,Complete secondary,Complete secondary,-0.655377
2,1001970838,PNC,PNC1,PNC1,1,17.833333,Male,Other,Hispanic or Latino,23.98,Right,11th Grade,Complete tertiary,Complete tertiary,-0.659061
3,100527940,PNC,PNC1,PNC1,1,8.250000,Male,Black,not Hispanic or Latino,NaN,Ambidextrous,1st Grade,Complete secondary,Complete primary,-0.591516
4,1006151876,PNC,PNC1,PNC1,1,21.500000,Female,Other,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,-0.377828


## 3. Data Loading & Feature Extraction Functions

In [4]:
def load_fsdata(participant_id, local_cache_dir=Path('./fs_data_cache')):
    """Download one participant's FreeSurfer region stats from RBC S3."""
    if local_cache_dir is not None:
        local_cache_dir = Path(local_cache_dir)
        local_cache_dir.mkdir(exist_ok=True)

    pnc_freesurfer_path = RBCPath(
        'rbc://PNC_FreeSurfer/freesurfer',
        local_cache_dir=local_cache_dir
    )
    tsv_path = (pnc_freesurfer_path / f'sub-{participant_id}'
                / f'sub-{participant_id}_regionsurfacestats.tsv')

    with tsv_path.open('r') as f:
        return pd.read_csv(f, sep='\t')


def extract_brain_features(subject_df):
    """Extract global mean thickness and total cortical volume (DKT atlas)."""
    dkt = subject_df[subject_df['atlas'] == 'aparc.DKTatlas']
    return {
        'global_mean_thickness': dkt['ThickAvg'].mean(),
        'total_cortical_volume': dkt['GrayVol'].sum()
    }


print("Functions defined.")

Functions defined.


## 4. Feature Engineering & Data Download

In [ ]:
all_data_eng = all_data.copy()

# Demographic engineering
all_data_eng['sex_numeric'] = all_data_eng['sex'].map({'Male': 0, 'Female': 1})
all_data_eng['age_sq'] = all_data_eng['age'] ** 2

edu_map = {level: i for i, level in enumerate(
    ['No/incomplete primary', 'Complete primary', 'Complete secondary', 'Complete tertiary']
)}
all_data_eng['parent_1_edu_numeric'] = all_data_eng['parent_1_education'].map(edu_map)
all_data_eng['parent_2_edu_numeric'] = all_data_eng['parent_2_education'].map(edu_map)
all_data_eng['parent_edu_avg'] = all_data_eng[
    ['parent_1_edu_numeric', 'parent_2_edu_numeric']
].mean(axis=1)

# Download brain features
all_vars = {col: [] for col in ['participant_id'] + FINAL_FEATURES + ['p_factor']}

prog = IntProgress(min=0, max=len(all_data_eng))
display(prog)
print("Downloading FreeSurfer data (cached after first run)...")

for _, row in all_data_eng.iterrows():
    pid = row['participant_id']
    try:
        feats = extract_brain_features(load_fsdata(pid))
    except (FileNotFoundError, HTTPError, KeyError):
        feats = {'global_mean_thickness': np.nan, 'total_cortical_volume': np.nan}

    all_vars['participant_id'].append(pid)
    all_vars['global_mean_thickness'].append(feats['global_mean_thickness'])
    all_vars['total_cortical_volume'].append(feats['total_cortical_volume'])
    all_vars['age'].append(row['age'])
    all_vars['age_sq'].append(row['age_sq'])
    all_vars['sex_numeric'].append(row['sex_numeric'])
    all_vars['parent_edu_avg'].append(row['parent_edu_avg'])
    all_vars['p_factor'].append(row['p_factor'])
    prog.value += 1

all_vars_df = pd.DataFrame(all_vars)

print(f"\nDone. {len(all_vars_df)} participants, "
      f"{all_vars_df['global_mean_thickness'].isna().sum()} missing brain data.")
all_vars_df.head()

IntProgress(value=0, max=1601)

## 6. Model: RidgeCV

In [ ]:
# Split
train_vars = all_vars_df[all_vars_df['p_factor'].notna()].copy()
test_vars = all_vars_df[all_vars_df['p_factor'].isna()].copy()

# Only drop rows missing target or core demographics (imputer handles brain NaNs)
complete_train = train_vars.dropna(subset=['age', 'age_sq', 'sex_numeric', 'parent_edu_avg', 'p_factor']).copy()

X_train = complete_train[FINAL_FEATURES].values
y_train = complete_train['p_factor'].values

print(f"Training matrix: {X_train.shape[0]} participants x {X_train.shape[1]} features")

# Pipeline
alphas = [0.1, 1.0, 10.0, 100.0]
pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), RidgeCV(alphas=alphas))

# Cross-validation across seeds
print(f"\n5-fold CV across seeds {STABILITY_SEEDS}:")
print("-" * 60)

seed_results = []
for seed in STABILITY_SEEDS:
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='r2')
    seed_results.append(np.mean(scores))
    print(f"  Seed {seed:2d} → R²: {np.mean(scores):.4f}  "
          f"(folds: {', '.join(f'{s:.4f}' for s in scores)})")

print("-" * 60)
print(f"Mean R²: {np.mean(seed_results):.4f} ± {np.std(seed_results):.4f}")

# Train final model on all training data
pipeline.fit(X_train, y_train)

print(f"\nSelected alpha: {pipeline.named_steps['ridgecv'].alpha_}")
print("\nStandardized coefficients:")
coefs = list(zip(FINAL_FEATURES, pipeline.named_steps['ridgecv'].coef_))
coefs.sort(key=lambda x: abs(x[1]), reverse=True)
for feat, w in coefs:
    print(f"  {feat:>25}: {w:+.4f}")

# Predict test set
X_test_raw = test_vars[FINAL_FEATURES].values
test_is_valid = ~np.isnan(X_test_raw).all(axis=1)  # at least one non-NaN feature

predicted_p = np.full(len(test_vars), y_train.mean())
if test_is_valid.sum() > 0:
    predicted_p[test_is_valid] = pipeline.predict(X_test_raw[test_is_valid])

print(f"\nTest predictions: range [{predicted_p.min():.3f}, {predicted_p.max():.3f}], "
      f"mean {predicted_p.mean():.3f}")

# Save
test_data_submit = test_data.copy()
test_data_submit['p_factor'] = predicted_p
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
test_data_submit.to_csv(OUTPUT_PATH, sep='\t', index=False)

print(f"\nSaved to: {OUTPUT_PATH}")